# Lab 17-09 · Path Loss, Multipath and Fading

Six parts, each built on `channels.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import special
from scipy.stats import norm
import channels as ch
rng = np.random.default_rng(1709)

## Part 1 · Two rays

*Predict:* the breakpoint for antennas at 10 m and 1.5 m at 3.5 GHz, and the slope of the loss beyond it.

In [2]:
f = 3.5e9
bp = ch.two_ray_breakpoint(10, 1.5, f)
print(f"breakpoint {bp:.1f} m")
d = np.logspace(1, 4, 800)
plt.semilogx(d, [ch.fspl_db(f, x) for x in d], label="free space")
plt.semilogx(d, [ch.two_ray_loss_db(x, 10, 1.5, f) for x in d], lw=0.8, label="two-ray")
plt.axvline(bp, ls=":", color="k"); plt.xlabel("distance (m)"); plt.ylabel("loss (dB)"); plt.legend(); plt.show()
slope = ch.two_ray_loss_db(1e4, 10, 1.5, f) - ch.two_ray_loss_db(1e3 * 2, 10, 1.5, f)
print(f"loss change from 2 km to 10 km: {slope:.1f} dB (40 log10 5 = {40 * math.log10(5):.1f} dB)")
assert abs(bp - 700.5) < 0.1

breakpoint 700.5 m
loss change from 2 km to 10 km: 27.5 dB (40 log10 5 = 28.0 dB)


## Part 2 · Coverage with shadowing

*Predict:* the fraction of NLOS users (σ = 7.82 dB) whose loss exceeds the median by less than 10 dB.

In [3]:
x = rng.normal(0, 7.82, 100_000)
frac = np.mean(x < 10)
print(f"simulated {frac:.4f}, predicted {norm.cdf(10 / 7.82):.4f}; 90% margin = {ch.shadow_margin_db(7.82, 0.9):.2f} dB")
print(f"UMi at 3.5 GHz, 100 m: LOS {ch.umi_los_db(100, 3.5):.2f} dB, NLOS {ch.umi_nlos_db(100, 3.5):.2f} dB")
assert abs(frac - norm.cdf(10 / 7.82)) < 0.005

simulated 0.8994, predicted 0.8995; 90% margin = 10.02 dB
UMi at 3.5 GHz, 100 m: LOS 85.31 dB, NLOS 104.64 dB


## Part 3 · Frequency selectivity

*Predict:* the frequency separation at which random channels with an exponential PDP (τ_rms = 100 ns) decorrelate to 0.5.

In [4]:
nch, npaths, tau = 2000, 200, 100e-9
taus = rng.exponential(tau, (nch, npaths))
g = (rng.standard_normal((nch, npaths)) + 1j * rng.standard_normal((nch, npaths))) / math.sqrt(2 * npaths)
dfs = np.linspace(0, 10e6, 41)
H0 = g.sum(axis=1)
corr = [abs(np.mean(H0 * np.conj((g * np.exp(-2j * np.pi * df * taus)).sum(axis=1)))) / np.mean(abs(H0) ** 2) for df in dfs]
plt.plot(dfs / 1e6, corr, "o", label="simulated")
plt.plot(dfs / 1e6, 1 / np.sqrt(1 + (2 * np.pi * dfs * tau) ** 2), label="1/sqrt(1 + (2 pi df tau)^2)")
plt.axhline(0.5, ls=":", color="k"); plt.xlabel("frequency separation (MHz)"); plt.ylabel("|correlation|"); plt.legend(); plt.show()
print(f"predicted 50% coherence bandwidth {ch.exp_pdp_coherence_bw(tau) / 1e6:.2f} MHz")
one = (g[0] * np.exp(-2j * np.pi * np.linspace(0, 20e6, 400)[:, None] * taus[0])).sum(axis=1)
plt.plot(np.linspace(0, 20, 400), 20 * np.log10(abs(one))); plt.xlabel("frequency (MHz)"); plt.ylabel("|H| (dB)"); plt.title("one channel across 20 MHz"); plt.show()

predicted 50% coherence bandwidth 2.76 MHz


## Part 4 · Doppler and level crossings

*Predict:* the autocorrelation at 2 ms for $f_D$ = 100 Hz, and the number and duration of fades below −10 dB per second.

In [5]:
fd, fs = 100.0, 20_000.0
h = np.concatenate([ch.rayleigh_sos(20_000, fd, fs, rng) for _ in range(20)])   # 20 s in 1 s segments
lags = np.arange(0, 200, 5)
acf = [np.mean([np.mean(seg[: len(seg) - L] * np.conj(seg[L:])).real for seg in np.split(h, 20)]) for L in lags]
plt.plot(lags / fs * 1e3, np.array(acf) / acf[0], "o", label="simulated")
plt.plot(lags / fs * 1e3, ch.jakes_corr(lags / fs, fd), label="J0(2 pi fd t)")
plt.xlabel("lag (ms)"); plt.ylabel("autocorrelation"); plt.legend(); plt.show()
rho = 10 ** (-10 / 20)
env = abs(h) / math.sqrt(np.mean(abs(h) ** 2))
below = env < rho
n_fades = np.sum(below[1:] & ~below[:-1]) / (len(h) / fs)
print(f"fades per second: simulated {n_fades:.1f}, formula {ch.lcr(rho, fd):.1f}")
print(f"mean fade duration: simulated {np.mean(below) / n_fades * 1e3:.2f} ms, formula {ch.afd(rho, fd) * 1e3:.2f} ms")
print(f"coherence time (0.5): {ch.coherence_time(fd) * 1e3:.2f} ms")

fades per second: simulated 71.2, formula 71.7
mean fade duration: simulated 1.34 ms, formula 1.33 ms
coherence time (0.5): 2.42 ms


## Part 5 · Outage

*Predict:* the probability of a fade more than 10 dB below the mean for Rayleigh fading and for Rician fading with K = 6 dB.

In [6]:
n = 500_000
ray = (rng.standard_normal(n) + 1j * rng.standard_normal(n)) / math.sqrt(2)
K = 10 ** 0.6
ric = math.sqrt(K / (K + 1)) + ray / math.sqrt(K + 1)
print(f"Rayleigh: simulated {np.mean(abs(ray) ** 2 < 0.1):.4f}, formula {ch.rayleigh_outage(10):.4f}")
print(f"Rician K = 6 dB: simulated {np.mean(abs(ric) ** 2 < 0.1):.4f}, formula {ch.rician_outage(10, K):.4f}")

Rayleigh: simulated 0.0944, formula 0.0952
Rician K = 6 dB: simulated 0.0164, formula 0.0165


## Part 6 · BER in fading with diversity

*Predict:* the BER at 15 dB for 1, 2 and 4 MRC branches.

In [7]:
grid = np.arange(0, 31, 5.0)
for L in (1, 2, 4):
    sims = []
    for x in grid:
        nb = 200_000
        bits = rng.integers(0, 2, nb)
        gains = (rng.standard_normal((L, nb)) + 1j * rng.standard_normal((L, nb))) / math.sqrt(2)
        noise = (rng.standard_normal((L, nb)) + 1j * rng.standard_normal((L, nb))) * math.sqrt(1 / (2 * 10 ** (x / 10)))
        y = gains * (1 - 2 * bits) + noise
        sims.append(np.mean((np.real(np.sum(np.conj(gains) * y, axis=0)) < 0) != bits))
    plt.semilogy(grid, np.maximum(sims, 1e-7), "o", label=f"L = {L} simulated")
    plt.semilogy(grid, [ch.ber_bpsk_mrc(10 ** (x / 10), L) for x in grid], "-")
    print(f"L = {L}: BER at 15 dB formula {ch.ber_bpsk_mrc(10 ** 1.5, L):.2e}")
plt.semilogy(grid, norm.sf(np.sqrt(2 * 10 ** (grid / 10))), "k--", label="AWGN")
plt.ylim(1e-7, 0.5); plt.xlabel("average Eb/N0 per branch (dB)"); plt.ylabel("BER"); plt.legend(); plt.show()

L = 1: BER at 15 dB formula 7.72e-03


L = 2: BER at 15 dB formula 1.78e-04


L = 4: BER at 15 dB formula 1.22e-07
